# A large scale analysis of conversation data across myriad spoken corpora

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [ ]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(bool(os.path.exists(FINAL_DOC)))

In [ ]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

#### Load Data

In [ ]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

In [ ]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

In [ ]:
df['lang'] = [lang(x) for x in tqdm(df['corpus'])]

I had some worries about using the japanese dataset with `xlm-roberta-base`. [Some other users have noted that xlm-roberta pathologically renders CoS (thus CoE values) superficially high (low)](https://huggingface.co/FacebookAI/xlm-roberta-base/discussions/18) when comparing across sentences (in other words, the model exhibits high anisotropy for Japanese in particular). 

It may be worth it to actually take this experiment and re-run all the vectors using a multilingual MPNET model. MPNET is architecturally similar to RoBERTa, but the cloze task was/is dependency aware, however. That may yield stronger interlanguage differences just by dint of it being a different model.

In [ ]:
df = df.loc[~df['lang'].isin([
    'jpn', 
    # 'fin', 'yid'
])]
df.shape

In [ ]:
df['conversation_id'].nunique()

In [ ]:
df.head()

## Convert values to numeric tags

In [ ]:
convert_to_numeric_id = [
    #'x_speaker', 'y_speaker', 
    # 'dyad', 
    'lang'
    #'x_speaker_turn'
]

In [ ]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}
    
    if isinstance(col,list):
        for c  in col:
            df[c] = [conversion[v] for v in tqdm(df[c])]
    
    else:
        df[col] = [conversion[v] for v in tqdm(df[col])]


### Other data preparation processes

In [ ]:
df.head()

## LME Regression: Predicting CE

In [ ]:
import statsmodels.formula.api as smf

In [ ]:
df = df.loc[~df['null']]

### Time-inclusive model

In [ ]:
##########################################
## Main model
##########################################
model = "Hxy ~ nx + ny + sample_delta*self + (1|lang)"
##########################################

In [ ]:
start = dt.now()
# md = smf.mixedlm(model, data=df, groups=df['x_speaker_turn'])
md = smf.mixedlm(model, data=df, groups=df['x_turn_id'])
mdf = md.fit()
print('completed in:', dt.now()-start)

In [ ]:
# mdf.summary()

In [ ]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

Saving results for reporting

In [ ]:
lme_results_name = os.path.join(DATA_LOC, 'lme-results-20250812.csv')

In [ ]:
reporting.to_csv(lme_results_name, encoding='utf-8')

In [ ]:
reporting['Var'] = reporting.index.values

with open(lme_results_name.replace('.csv', '.txt'), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()